# 📚 Clasificación del Lenguaje SQL
## Por: Kimi (Asistente de IA - Moonshot AI)

SQL se divide en sublenguajes según su función:

---

### 🔧 DDL - Data Definition Language
**Para:** Crear, modificar, eliminar **estructuras** de la base de datos.

| Comando | Función |
|---------|---------|
| CREATE | Crear tablas, índices |
| ALTER | Modificar tablas existentes |
| DROP | Eliminar tablas |

**Ejemplo:** `CREATE TABLE clientes (id INTEGER, nombre TEXT)`

---

### 📝 DML - Data Manipulation Language
**Para:** Trabajar con los **datos** dentro de las tablas.

| Comando | Función |
|---------|---------|
| SELECT | Consultar datos |
| INSERT | Agregar registros |
| UPDATE | Modificar registros |
| DELETE | Eliminar registros |

**Ejemplo:** `SELECT * FROM clientes WHERE nombre = 'Juan'`

---

### 🔐 DCL - Data Control Language
**Para:** Controlar **permisos** de acceso.

| Comando | Función |
|---------|---------|
| GRANT | Dar permisos |
| REVOKE | Quitar permisos |

**Ejemplo:** `GRANT SELECT ON clientes TO usuario1`

---

### ⚡ TCL - Transaction Control Language
**Para:** Gestionar **transacciones** (grupos de operaciones).

| Comando | Función |
|---------|---------|
| COMMIT | Confirmar cambios |
| ROLLBACK | Deshacer cambios |

---

## 🎯 Este Tutorial Incluye:

| Pestaña | Tipo SQL | Función |
|---------|----------|---------|
| 🏗️ **Administrar** | DDL | Crear/ver/eliminar tablas |
| 💻 **SQL Libre** | DML | Escribir consultas directamente |
| 🔍 **Consultar** | DML | Filtros visuales simples |
| ➕ **Agregar** | DML | Insertar registros con formulario |

&gt; **Autoría:** Kimi (Moonshot AI) - Febrero 2025
&gt; **Destinado a:** Alumnos de primer semestre de Ingeniería

## Paso 1: Importar Librerías

Necesitamos 4 herramientas básicas:

- **sqlite3**: Base de datos (viene con Python)
- **pandas**: Mostrar datos en tablas bonitas
- **ipywidgets**: Crear botones y formularios
- **IPython.display**: Mostrar elementos en pantalla

Todas están disponibles en JupyterLite sin instalación.

In [1]:
import sqlite3
import pandas as pd
import os  # Para Sistema Operativo
import micropip
await micropip.install('ipywidgets')
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import base64

print("✅ Librerías cargadas")
print(f"SQLite: {sqlite3.sqlite_version}")

✅ Librerías cargadas
SQLite: 3.39.0


## Paso 2: Conectar a la Base de Datos

Creamos la conexión al archivo `sistema.db` y definimos funciones auxiliares que usaremos en toda la aplicación:

- `ejecutar_sql()`: Ejecuta cualquier comando SQL y muestra resultados
- `listar_tablas()`: Muestra todas las tablas existentes
- `obtener_datos()`: Lee datos de una tabla como DataFrame de pandas

In [2]:
# ============================================
# CELDA 2 - CONEXIÓN ROBUSTA (MEMORIA + RESPALDO)
# ============================================

import os
import sqlite3
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import base64
import json

# ============================================
# ESTRATEGIA: Base de datos en MEMORIA (estable)
# con opción de exportar/importar a archivo
# ============================================

print("🔄 Inicializando sistema de base de datos...")

# Usar :memory: para evitar corrupción de archivo en JupyterLite
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

print("✅ Base de datos en MEMORIA creada (estable)")

# ============================================
# CREAR TABLA CUENTAS
# ============================================

cursor.execute('''
    CREATE TABLE cuentas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente TEXT,
        monto REAL,
        estado TEXT DEFAULT 'Pendiente'
    )
''')
print("✅ Tabla 'cuentas' creada")

# ============================================
# INSERTAR DATOS DE PRUEBA
# ============================================

datos_demo = [
    ('Ana Martínez', 2500.00, 'Pendiente'),
    ('Luis Torres', 1800.50, 'Pagado'),
    ('Carmen Ruiz', 3200.00, 'Pendiente'),
    ('Pedro Gómez', 950.00, 'Pagado'),
    ('María López', 4100.00, 'Pendiente'),
    ('Juan Sánchez', 2750.75, 'Pendiente'),
]

try:
    for cliente, monto, estado in datos_demo:
        cursor.execute(
            "INSERT INTO cuentas (cliente, monto, estado) VALUES (?, ?, ?)",
            (cliente, monto, estado)
        )
    print(f"✅ {len(datos_demo)} registros insertados")
except Exception as e:
    print(f"❌ Error al insertar: {e}")
    raise

# Verificar
cursor.execute("SELECT COUNT(*) FROM cuentas")
count = cursor.fetchone()[0]
print(f"📊 Verificación: {count} registros en tabla")

# ============================================
# FUNCIONES AUXILIARES
# ============================================

def ejecutar_sql(comando):
    """Ejecuta SQL y devuelve DataFrame para SELECT, None para otros, o error"""
    try:
        cursor.execute(comando)
        if comando.strip().upper().startswith('SELECT'):
            return pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
        return None
    except Exception as e:
        return f"❌ Error: {e}"

def listar_tablas():
    """Lista tablas en la base de datos en memoria"""
    try:
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
        return [r[0] for r in cursor.fetchall()]
    except Exception as e:
        print(f"⚠️ Error al listar: {e}")
        return []

def obtener_datos(tabla):
    """Obtiene datos de una tabla como DataFrame"""
    try:
        return pd.read_sql(f"SELECT * FROM {tabla}", conn)
    except:
        return pd.DataFrame()

def tabla_existe(nombre):
    """Verifica si tabla existe"""
    return nombre in listar_tablas()

# ============================================
# FUNCIÓN DE RESPALDO (MEMORIA -> ARCHIVO)
# ============================================

def respaldar_a_archivo(nombre_archivo='sistema_respaldo.db'):
    """
    Copia la base de datos en memoria a un archivo.
    Útil para guardar trabajo antes de cerrar JupyterLite.
    """
    try:
        # Crear conexión a archivo
        disk_conn = sqlite3.connect(nombre_archivo)
        
        # Copiar esquema y datos
        for linea in conn.iterdump():
            disk_conn.execute(linea)
        
        disk_conn.commit()
        disk_conn.close()
        return True, nombre_archivo
    except Exception as e:
        return False, str(e)

def restaurar_desde_archivo(nombre_archivo='sistema_respaldo.db'):
    """
    Restaura base de datos desde archivo a memoria.
    """
    global conn, cursor
    
    if not os.path.exists(nombre_archivo):
        return False, "Archivo no encontrado"
    
    try:
        # Cerrar memoria actual
        conn.close()
        
        # Crear nueva conexión en memoria
        conn = sqlite3.connect(':memory:')
        cursor = conn.cursor()
        
        # Leer archivo y ejecutar en memoria
        disk_conn = sqlite3.connect(nombre_archivo)
        for linea in disk_conn.iterdump():
            cursor.execute(linea)
        
        disk_conn.close()
        conn.commit()
        return True, "Restaurado correctamente"
    except Exception as e:
        return False, str(e)

print(f"📋 Tablas disponibles: {listar_tablas()}")

# ============================================
# WIDGETS DDL
# ============================================

ddl_nombre = widgets.Text(description='Nombre:', placeholder='nombre_tabla')
ddl_columnas = widgets.Textarea(
    description='Columnas:',
    placeholder='id INTEGER PRIMARY KEY, nombre TEXT, precio REAL',
    layout=widgets.Layout(height='60px')
)
ddl_boton_crear = widgets.Button(description='Crear Tabla', button_style='success', icon='plus')

# Inicializar dropdown con tablas existentes
tablas_iniciales = listar_tablas()
ddl_tabla_sel = widgets.Dropdown(
    description='Tabla:', 
    options=tablas_iniciales,
    value=tablas_iniciales[0] if tablas_iniciales else None
)

ddl_boton_ver = widgets.Button(description='Ver Estructura', button_style='info', icon='search')
ddl_boton_borrar = widgets.Button(description='Eliminar', button_style='danger', icon='trash')
ddl_boton_actualizar = widgets.Button(description='🔄', layout=widgets.Layout(width='50px'))

ddl_sql = widgets.Textarea(
    description='SQL DDL:',
    placeholder='CREATE TABLE ... / DROP TABLE ...',
    layout=widgets.Layout(height='80px')
)
ddl_boton_ejecutar = widgets.Button(description='Ejecutar SQL', button_style='warning', icon='play')

area_ddl = widgets.Output()

# ============================================
# FUNCIONES DDL
# ============================================

TABLA_PROTEGIDA = 'cuentas'

def actualizar_dropdown_tablas():
    """Actualiza dropdown con tablas actuales"""
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tablas = [r[0] for r in cursor.fetchall()]
    
    ddl_tabla_sel.options = tablas
    if tablas and (ddl_tabla_sel.value not in tablas or ddl_tabla_sel.value is None):
        ddl_tabla_sel.value = tablas[0]
    elif not tablas:
        ddl_tabla_sel.value = None
    return tablas

def crear_tabla(b):
    with area_ddl:
        clear_output()
        
        if not ddl_nombre.value.strip():
            print("❌ Escribe nombre de tabla")
            return
        
        nombre = ddl_nombre.value.strip().replace(' ', '_')
        if not all(c.isalnum() or c == '_' for c in nombre):
            print("❌ Nombre inválido (solo letras, números, guiones bajos)")
            return
        
        if nombre == TABLA_PROTEGIDA and tabla_existe(nombre):
            print(f"⚠️ La tabla '{nombre}' ya existe")
            return
            
        sql = f"CREATE TABLE {nombre} ({ddl_columnas.value})"
        resultado = ejecutar_sql(sql)
        
        if resultado is None:
            print(f"✅ Tabla '{nombre}' creada")
            tablas = actualizar_dropdown_tablas()
            ddl_tabla_sel.value = nombre
            print(f"📋 Tablas: {tablas}")
            ddl_nombre.value = ''
            ddl_columnas.value = ''
        else:
            print(f"❌ Error: {resultado}")

def ver_estructura(b):
    with area_ddl:
        clear_output()
        
        if not ddl_tabla_sel.value:
            print("❌ Selecciona una tabla")
            return
            
        try:
            cols = cursor.execute(f"PRAGMA table_info({ddl_tabla_sel.value})").fetchall()
            print(f"📋 Estructura de '{ddl_tabla_sel.value}':")
            for c in cols:
                pk = " 🔑 PK" if c[5] else ""
                print(f"  {c[1]} ({c[2]}){pk}")
        except Exception as e:
            print(f"❌ Error: {e}")

def eliminar_tabla(b):
    with area_ddl:
        clear_output()
        
        if not ddl_tabla_sel.value:
            print("❌ Selecciona una tabla")
            return
        
        nombre = ddl_tabla_sel.value
        
        if nombre == TABLA_PROTEGIDA:
            print(f"⛔ No se puede eliminar '{nombre}'")
            return
        
        ejecutar_sql(f"DROP TABLE {nombre}")
        print(f"🗑️ Tabla '{nombre}' eliminada")
        actualizar_dropdown_tablas()

def ejecutar_ddl(b):
    with area_ddl:
        clear_output()
        
        if not ddl_sql.value.strip():
            print("❌ Escribe SQL")
            return
        
        # Bloquear DROP de tabla protegida
        cmd = ddl_sql.value.upper()
        if 'DROP' in cmd and 'TABLE' in cmd and TABLA_PROTEGIDA.upper() in cmd:
            print(f"⛔ Protegido: no se puede eliminar '{TABLA_PROTEGIDA}'")
            return
        
        resultado = ejecutar_sql(ddl_sql.value)
        
        if resultado is None:
            print("✅ Comando ejecutado")
            actualizar_dropdown_tablas()
        else:
            print(resultado)

def actualizar_desde_boton(b):
    with area_ddl:
        clear_output()
        tablas = actualizar_dropdown_tablas()
        print(f"🔄 Tablas: {tablas}")

# CONEXIONES
ddl_boton_crear.on_click(crear_tabla)
ddl_boton_ver.on_click(ver_estructura)
ddl_boton_borrar.on_click(eliminar_tabla)
ddl_boton_actualizar.on_click(actualizar_desde_boton)
ddl_boton_ejecutar.on_click(ejecutar_ddl)

print("=" * 50)
print("✅ CELDA 2 COMPLETADA")
print("=" * 50)
print("Base de datos: MEMORIA (sin corrupción)")
print(f"Tablas iniciales: {listar_tablas()}")
print("Funciones: ejecutar_sql, listar_tablas, obtener_datos, tabla_existe")
print("Widgets DDL: creados y listos")
print("=" * 50)

🔄 Inicializando sistema de base de datos...
✅ Base de datos en MEMORIA creada (estable)
✅ Tabla 'cuentas' creada
✅ 6 registros insertados
📊 Verificación: 6 registros en tabla
📋 Tablas disponibles: ['cuentas']
✅ CELDA 2 COMPLETADA
Base de datos: MEMORIA (sin corrupción)
Tablas iniciales: ['cuentas']
Funciones: ejecutar_sql, listar_tablas, obtener_datos, tabla_existe
Widgets DDL: creados y listos


## Paso 3: Crear Tabla Inicial

Si no existe ninguna tabla, creamos automáticamente `cuentas` con campos típicos de un sistema de cobranza:

- **id**: Número único automático
- **cliente**: Nombre del deudor
- **monto**: Cantidad a pagar
- **estado**: Pendiente o Pagado

Esto permite empezar a usar el sistema inmediatamente.

In [3]:
# ============================================
# CELDA 3 - SQL LIBRE
# ============================================

# WIDGETS SQL LIBRE
sql_libre = widgets.Textarea(
    description='SQL:',
    placeholder='SELECT * FROM cuentas WHERE monto > 1000\n-- O: INSERT, UPDATE, DELETE, CREATE...',
    layout=widgets.Layout(width='100%', height='120px')
)

sql_boton_ejecutar = widgets.Button(
    description='Ejecutar', 
    button_style='primary', 
    icon='play'
)

sql_boton_limpiar = widgets.Button(
    description='Limpiar', 
    button_style='warning', 
    icon='eraser'
)

area_sql = widgets.Output()

def ejecutar_sql_libre(b):
    with area_sql:
        clear_output()
        
        if not sql_libre.value.strip():
            print("❌ Escribe una consulta SQL")
            return
        
        comando = sql_libre.value.strip()
        resultado = ejecutar_sql(comando)
        
        if resultado is None:
            print("✅ Comando ejecutado correctamente")
            print(f"Tipo: {comando.split()[0].upper()}")
            # Actualizar consultas si existe
            try:
                mostrar_consultas()
            except:
                pass
        elif isinstance(resultado, str) and resultado.startswith('❌'):
            print(resultado)
        else:
            print(f"📊 Resultados ({len(resultado)} filas):")
            display(resultado)

def limpiar_sql(b):
    sql_libre.value = ''
    with area_sql:
        clear_output()
        print("🧹 Área limpiada")

sql_boton_ejecutar.on_click(ejecutar_sql_libre)
sql_boton_limpiar.on_click(limpiar_sql)

print("✅ Pestaña SQL Libre lista")

✅ Pestaña SQL Libre lista


In [4]:
# ============================================
# CELDA 4 - CONSULTAS VISUALES + REGEX
# ============================================

import re

# WIDGETS CONSULTAS VISUALES
cons_tabla_sel = widgets.Dropdown(
    description='Tabla:', 
    options=listar_tablas(),
    layout=widgets.Layout(width='200px')
)

cons_cliente = widgets.Text(
    description='Cliente:', 
    placeholder='Nombre o parte',
    layout=widgets.Layout(width='300px')
)

cons_estado = widgets.Dropdown(
    description='Estado:', 
    options=['Cualquiera', 'Pendiente', 'Pagado'],
    layout=widgets.Layout(width='200px')
)

cons_monto = widgets.FloatText(
    description='Monto mín.:', 
    value=0,
    layout=widgets.Layout(width='200px')
)

cons_boton_actualizar = widgets.Button(
    description='🔄', 
    layout=widgets.Layout(width='50px'),
    tooltip='Actualizar lista de tablas'
)

cons_boton_exportar = widgets.Button(
    description='Exportar CSV', 
    button_style='info', 
    icon='download'
)

area_consultas = widgets.Output()

# WIDGETS REGEX
regex_campo = widgets.Dropdown(
    description='Campo:',
    options=['cliente', 'estado'],
    value='cliente',
    layout=widgets.Layout(width='200px')
)

regex_patron = widgets.Text(
    description='Patrón regex:',
    placeholder='Ej: ^A.* o P.*',
    layout=widgets.Layout(width='400px')
)

regex_boton = widgets.Button(
    description='Buscar con Regex',
    button_style='warning',
    icon='search'
)

regex_ayuda = widgets.HTML(
    value="<small><b>Ejemplos:</b> <code>^A</code> (empieza con A) | <code>z$</code> (termina en z)</small>"
)

area_regex = widgets.Output()

# FUNCIÓN DE FILTRADO
def mostrar_consultas(cambio=None):
    with area_consultas:
        clear_output()
        
        tablas_disponibles = listar_tablas()
        if not tablas_disponibles:
            print("⚠️ No hay tablas disponibles")
            return
        
        # Sincronizar dropdown si cambió
        if set(cons_tabla_sel.options) != set(tablas_disponibles):
            cons_tabla_sel.options = tablas_disponibles
        
        tabla = cons_tabla_sel.value
        if not tabla:
            print("❌ Selecciona una tabla")
            return
        
        df = obtener_datos(tabla)
        
        if df.empty:
            print(f"ℹ️ La tabla '{tabla}' está vacía")
            return
        
        df_filtrado = df.copy()
        
        try:
            if cons_cliente.value and 'cliente' in df_filtrado.columns:
                df_filtrado = df_filtrado[
                    df_filtrado['cliente'].astype(str).str.contains(
                        cons_cliente.value, case=False, na=False
                    )
                ]
            
            if cons_estado.value != 'Cualquiera' and 'estado' in df_filtrado.columns:
                df_filtrado = df_filtrado[df_filtrado['estado'] == cons_estado.value]
            
            if cons_monto.value > 0 and 'monto' in df_filtrado.columns:
                df_filtrado = df_filtrado[df_filtrado['monto'] >= cons_monto.value]
                
        except Exception as e:
            print(f"⚠️ Error en filtros: {e}")
        
        print(f"📊 Tabla: '{tabla}' | {len(df_filtrado)} de {len(df)} registro(s)")
        print("-" * 50)
        
        if df_filtrado.empty:
            print("🔍 No hay registros con esos filtros")
        else:
            display(df_filtrado)

# FUNCIÓN REGEX
def buscar_regex(b):
    with area_regex:
        clear_output()
        
        if cons_tabla_sel.value != 'cuentas':
            print("⚠️ Regex solo disponible para tabla 'cuentas'")
            return
        
        if not regex_patron.value:
            print("❌ Escribe un patrón regex")
            return
        
        df = obtener_datos('cuentas')
        campo = regex_campo.value
        patron = regex_patron.value
        
        try:
            mascara = df[campo].astype(str).str.match(patron, case=False)
            resultado = df[mascara]
            
            print(f"🔍 Regex: /{patron}/ en '{campo}'")
            print(f"📊 {len(resultado)} coincidencias de {len(df)}")
            
            if resultado.empty:
                print("❌ No hay coincidencias")
            else:
                display(resultado)
                
        except re.error as e:
            print(f"❌ Error en patrón: {e}")

def actualizar_tablas_consulta(b=None):
    tablas = listar_tablas()
    cons_tabla_sel.options = tablas
    with area_consultas:
        clear_output()
        print(f"🔄 Tablas: {tablas}")

def exportar_csv(b):
    with area_consultas:
        clear_output()
        
        tabla = cons_tabla_sel.value
        if not tabla or not tabla_existe(tabla):
            print("❌ Selecciona una tabla válida")
            return
        
        df = obtener_datos(tabla)
        
        # Aplicar filtros
        try:
            if cons_cliente.value and 'cliente' in df.columns:
                df = df[df['cliente'].astype(str).str.contains(cons_cliente.value, case=False, na=False)]
            if cons_estado.value != 'Cualquiera' and 'estado' in df.columns:
                df = df[df['estado'] == cons_estado.value]
            if cons_monto.value > 0 and 'monto' in df.columns:
                df = df[df['monto'] >= cons_monto.value]
        except:
            pass
        
        if df.empty:
            print("⚠️ No hay datos para exportar")
            return
        
        try:
            csv = df.to_csv(index=False)
            b64 = base64.b64encode(csv.encode()).decode()
            nombre = f"{tabla}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            
            display(HTML(f'''
                <a download="{nombre}" href="data:text/csv;base64,{b64}" 
                   style="padding:10px; background:#2196F3; color:white; 
                          text-decoration:none; border-radius:4px;">
                   📥 {nombre}
                </a>
            '''))
        except Exception as e:
            print(f"❌ Error: {e}")

# CONEXIONES
cons_tabla_sel.observe(mostrar_consultas, names='value')
cons_cliente.observe(mostrar_consultas, names='value')
cons_estado.observe(mostrar_consultas, names='value')
cons_monto.observe(mostrar_consultas, names='value')
cons_boton_actualizar.on_click(actualizar_tablas_consulta)
cons_boton_exportar.on_click(exportar_csv)
regex_boton.on_click(buscar_regex)

print("✅ Pestaña consultas visuales lista")

✅ Pestaña consultas visuales lista


In [5]:
# ============================================
# CELDA 5 - AGREGAR REGISTROS
# ============================================

# WIDGETS AGREGAR
add_cliente = widgets.Text(description='Cliente:', placeholder='Nombre completo')
add_monto = widgets.FloatText(description='Monto:', value=0, min=0)
add_estado = widgets.Dropdown(description='Estado:', options=['Pendiente', 'Pagado'], value='Pendiente')
add_boton = widgets.Button(description='Guardar', button_style='success', icon='check')
area_add = widgets.Output()

def insertar_registro(b):
    with area_add:
        clear_output()
        
        if not tabla_existe('cuentas'):
            print("❌ Error: La tabla 'cuentas' no existe")
            return
        
        cliente = add_cliente.value.strip()
        if not cliente:
            print("❌ Cliente es obligatorio")
            return
            
        if add_monto.value <= 0:
            print("❌ Monto debe ser mayor a cero")
            return
        
        try:
            cursor.execute(
                "INSERT INTO cuentas (cliente, monto, estado) VALUES (?, ?, ?)",
                (cliente, float(add_monto.value), add_estado.value)
            )
            
            nuevo_id = cursor.lastrowid
            
            print(f"✅ Registro guardado (ID: {nuevo_id})")
            print(f"   Cliente: {cliente}")
            print(f"   Monto: ${add_monto.value:,.2f}")
            
            # Limpiar
            add_cliente.value = ''
            add_monto.value = 0
            add_estado.value = 'Pendiente'
            
            # Actualizar otras pestañas
            try:
                mostrar_consultas()
            except:
                pass
                
        except Exception as e:
            print(f"❌ Error: {e}")

add_boton.on_click(insertar_registro)

print("✅ Pestaña agregar registros lista")

✅ Pestaña agregar registros lista


In [6]:
# ============================================
# CELDA 6 - RESPALDO TOTAL
# ============================================

respaldo_boton = widgets.Button(
    description='💾 Respaldo Total (CSV)',
    button_style='success',
    icon='download'
)

area_respaldo = widgets.Output()

def respaldo_total_csv(b):
    with area_respaldo:
        clear_output()
        
        tablas = listar_tablas()
        
        if not tablas:
            print("⚠️ No hay tablas para respaldar")
            return
        
        print(f"📦 Respaldando {len(tablas)} tabla(s)...")
        print("-" * 50)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        for tabla in tablas:
            try:
                df = obtener_datos(tabla)
                
                if df.empty:
                    print(f"⚠️  '{tabla}': vacía")
                    continue
                
                csv = df.to_csv(index=False)
                b64 = base64.b64encode(csv.encode()).decode()
                nombre = f"{tabla}_{timestamp}.csv"
                
                display(HTML(f'''
                    <a download="{nombre}" href="data:text/csv;base64,{b64}" 
                       style="padding:8px; background:#4CAF50; color:white; 
                              text-decoration:none; border-radius:4px; 
                              display:inline-block; margin:3px; font-size:12px;">
                       📥 {nombre} ({len(df)} filas)
                    </a>
                '''))
                
                print(f"✅ '{tabla}': {len(df)} filas")
                
            except Exception as e:
                print(f"❌ '{tabla}': {e}")
        
        print("-" * 50)
        print(f"🎉 Respaldo completado")

respaldo_boton.on_click(respaldo_total_csv)

print("✅ Función de respaldo lista")

✅ Función de respaldo lista


In [7]:
# ============================================
# CELDA 7 - ENSAMBLAR 4 PESTAÑAS
# ============================================

# PESTAÑA 0: ADMINISTRAR (DDL)
pestaña_ddl = widgets.VBox([
    widgets.HTML("<h3>⚙️ Administrar Tablas (DDL)</h3>"),
    widgets.HTML("<b>Crear nueva tabla:</b>"),
    widgets.HBox([ddl_nombre, ddl_columnas]),
    ddl_boton_crear,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Gestionar tablas:</b>"),
    widgets.HBox([ddl_tabla_sel, ddl_boton_ver, ddl_boton_borrar, ddl_boton_actualizar]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Respaldo:</b>"),
    respaldo_boton,
    area_respaldo,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>SQL DDL libre:</b>"),
    ddl_sql,
    ddl_boton_ejecutar,
    area_ddl
])

# PESTAÑA 1: SQL LIBRE
pestaña_sql = widgets.VBox([
    widgets.HTML("<h3>💻 Modo SQL Libre</h3>"),
    sql_libre,
    widgets.HBox([sql_boton_ejecutar, sql_boton_limpiar]),
    area_sql
])

# PESTAÑA 2: CONSULTAR
pestaña_consulta = widgets.VBox([
    widgets.HTML("<h3>🔍 Consultas Visuales</h3>"),
    widgets.HBox([cons_tabla_sel, cons_boton_actualizar]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Filtros:</b>"),
    widgets.HBox([cons_cliente, cons_estado, cons_monto]),
    cons_boton_exportar,
    area_consultas,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Regex (solo 'cuentas'):</b>"),
    widgets.HBox([regex_campo, regex_patron]),
    regex_ayuda,
    regex_boton,
    area_regex
])

# PESTAÑA 3: AGREGAR
pestaña_agregar = widgets.VBox([
    widgets.HTML("<h3>➕ Agregar Registro</h3>"),
    add_cliente, 
    add_monto, 
    add_estado,
    add_boton,
    area_add
])

# CREAR TABS
tabs = widgets.Tab(children=[
    pestaña_ddl, 
    pestaña_sql, 
    pestaña_consulta, 
    pestaña_agregar
])

tabs.set_title(0, '⚙️ Administrar')
tabs.set_title(1, '💻 SQL Libre')
tabs.set_title(2, '🔍 Consultar')
tabs.set_title(3, '➕ Agregar')

display(tabs)

# Cargar datos iniciales
try:
    mostrar_consultas()
except:
    pass

print("\n" + "=" * 60)
print("✅ SISTEMA LISTO - 4 PESTAÑAS ACTIVAS")
print("=" * 60)


✅ SISTEMA LISTO - 4 PESTAÑAS ACTIVAS


# Ejemplo crear tablas
En pestaña **Administrar**<br>
## Ejemplo 1:
| Campo        | Valor                                                                                   |
| ------------ | --------------------------------------------------------------------------------------- |
| **Nombre**   | `ventas`                                                                                |
| **Columnas** | `id INTEGER PRIMARY KEY, producto_id INTEGER, cantidad INTEGER, fecha TEXT, total REAL` |

## Ejemplo 2: 
| Campo        | Valor                                                                                                         |
| ------------ | ------------------------------------------------------------------------------------------------------------- |
| **Nombre**   | `empleados`                                                                                                   |
| **Columnas** | `id INTEGER PRIMARY KEY, nombre TEXT, puesto TEXT, salario REAL, contratacion DATE, activo INTEGER DEFAULT 1` |


# Ejemplo en SQL Libre
```
INSERT INTO productos (nombre, precio, stock) VALUES ('Laptop Dell', 15000.00, 10);
```   
```    
INSERT INTO productos (nombre, precio, stock) VALUES 
('Mouse inalámbrico', 350.50, 50),
('Teclado mecánico', 1200.00, 25),
('Monitor 27 pulgadas', 4500.00, 8),
('USB 64GB', 280.00, 100);<br>
```
```
SELECT * FROM productos;
```
```
SELECT * FROM productos WHERE precio > 1000;
```

```
UPDATE productos SET stock = 5 WHERE nombre = 'Laptop Dell';
```

```
DELETE FROM productos WHERE stock = 0;
```